## Self-Supervised Speech Recognition (data2vec)

In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
from pathlib import Path
import pandas as pd
import torch
import librosa
from transformers import (
  AutoProcessor,
  Data2VecAudioModel,
  Data2VecAudioForCTC,
  Wav2Vec2CTCTokenizer,
  Wav2Vec2FeatureExtractor,
  Wav2Vec2Processor,
)
from src.dataset import TorontoDataset, DataCollatorCTCWithPadding, sanitize
from torch.utils.data import DataLoader
import IPython.display as ipd
import lightning as L
from lightning.pytorch.loggers import CSVLogger
import evaluate
import os
import json

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
DATA_DIR = Path("data")
TORONTO = DATA_DIR / Path("toronto")
TIMIT = DATA_DIR / Path("timit")
TORONTO_SR = 44100
TIMIT_SR = 16000

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

In [24]:
labels = pd.read_json(TORONTO / "labels.jsonl", lines=True)

labels["dataset/toronto_0/toronto_0_0.wav"][0]

'Терміново! Загроза нового вторгнення в Україну нині дуже висока.'

In [25]:
toronto_ids = set([f.name for f in TORONTO.glob("toronto_*")])

toronto_test_ids = set(['toronto_27', 'toronto_46', 'toronto_42', 'toronto_37', 'toronto_89', 'toronto_43',
'toronto_157', 'toronto_9', 'toronto_156', 'toronto_7', 'toronto_123', 'toronto_54', 'toronto_67',
'toronto_62', 'toronto_81', 'toronto_134', 'toronto_148', 'toronto_21', 'toronto_135', 'toronto_166',
'toronto_58'])

toronto_df = []
for f in TORONTO.rglob("*.wav"):
    toronto_df.append({
        "filename": f.name,
        "path": str(f),
        "speaker_id": f.parent.name,
        "duration": librosa.get_duration(path=f),
        "label": labels[f"dataset/{f.parent.name}/{f.name}"][0]
    })

toronto_df = pd.DataFrame(toronto_df)
toronto_df = sanitize(toronto_df)

toronto_train_df = toronto_df[toronto_df["speaker_id"].isin(toronto_ids - toronto_test_ids)]
toronto_test_df = toronto_df[toronto_df["speaker_id"].isin(toronto_test_ids)]

After filter: 12827 / 18303 rows


In [26]:
BASELINE_MODEL = "Respeecher/ukrainian-data2vec-asr"

baseline_processor = AutoProcessor.from_pretrained(BASELINE_MODEL)
baseline_model = Data2VecAudioForCTC.from_pretrained(BASELINE_MODEL)

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

In [27]:
baseline_model.eval()

sample = toronto_test_df.iloc[0]

audio, sr = librosa.load(sample.path, sr=16_000)

print(sample)

inputs = baseline_processor(audio, sampling_rate=sr, return_tensors="pt")

with torch.no_grad():
    logits = baseline_model(**inputs).logits

predicted_ids = torch.argmax(logits, dim=-1)

transcription = baseline_processor.batch_decode(predicted_ids)

print(f"Original:\t{sample.label}")
print(f"Transcript:\t{transcription[0]}")

ipd.Audio(sample.path)

filename                                      toronto_62_13.wav
path                  data/toronto/toronto_62/toronto_62_13.wav
speaker_id                                           toronto_62
duration                                                    4.7
label         зародились протожарти про схожість звуків які ...
Name: 976, dtype: object
Original:	зародились протожарти про схожість звуків які були ознакою розвитку мовлення та писемності
Transcript:	зродились протожарти про схожість звуків які були ознакою розвідку вмовлення та писемності


In [ ]:
collator = DataCollatorCTCWithPadding(processor=baseline_processor)

toronto_train_ds = TorontoDataset(toronto_train_df, baseline_processor, TORONTO, features_column="input_values")
toronto_val_ds = TorontoDataset(toronto_test_df, baseline_processor, TORONTO, features_column="input_values")

toronto_train_dl = DataLoader(toronto_train_ds, batch_size=8, shuffle=True, collate_fn=collator)
toronto_val_dl = DataLoader(toronto_val_ds, batch_size=8, shuffle=False, collate_fn=collator)

In [29]:
class Data2VecCTCBaseline(L.LightningModule):
  def __init__(self, model, processor):
    super().__init__()
    self.model = model
    self.processor = processor
    self.wer = evaluate.load("wer")
    self.cer = evaluate.load("cer")

  def training_step(self, batch, _):
    loss = self.model(**batch).loss
    self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def validation_step(self, batch, _):
    out = self.model(**batch)
    self.log("val_loss", out.loss, prog_bar=True)

    pred_ids = torch.argmax(out.logits, dim=-1)

    labels = batch["labels"].clone()
    labels[labels == -100] = self.processor.tokenizer.pad_token_id

    preds = self.processor.batch_decode(pred_ids)
    refs = self.processor.batch_decode(labels, group_tokens=False)

    self.wer.add_batch(predictions=preds, references=refs)
    self.cer.add_batch(predictions=preds, references=refs)

  def on_validation_epoch_end(self):
    self.log("val_wer", self.wer.compute(), prog_bar=True)
    self.log("val_cer", self.cer.compute(), prog_bar=True)

  def configure_optimizers(self):
    return torch.optim.AdamW(self.model.parameters(), lr=1e-5)

In [30]:
baseline_collator = DataCollatorCTCWithPadding(
    processor=baseline_processor,
)

toronto_val_dl = DataLoader(toronto_val_ds, batch_size=8, shuffle=False, collate_fn=baseline_collator)

logger = CSVLogger("logs", name="data2vec_baseline_val")
baseline = Data2VecCTCBaseline(baseline_model, baseline_processor)

trainer = L.Trainer(accelerator=device, logger=logger)
# trainer.validate(baseline, dataloaders=toronto_val_dl)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [31]:
BACKBONE_MODEL = "Respeecher/ukrainian-data2vec"

baseline_processor = AutoProcessor.from_pretrained(BASELINE_MODEL)
baseline_model = Data2VecAudioModel.from_pretrained(BASELINE_MODEL)

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Data2VecAudioModel LOAD REPORT from: Respeecher/ukrainian-data2vec-asr
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 
lm_head.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
VOCAB = TORONTO / "vocab.json"

all_text = " ".join(toronto_train_df["label"])
vocab = sorted(set(all_text))
vocab_dict = {v: i for i, v in enumerate(vocab)}
vocab_dict["|"] = vocab_dict.pop(" ")
vocab_dict["<unk>"] = len(vocab_dict)
vocab_dict["<pad>"] = len(vocab_dict)

with open(VOCAB, "w", encoding="utf-8") as f:
    json.dump(vocab_dict, f, ensure_ascii=False)

In [33]:
tokenizer = Wav2Vec2CTCTokenizer(VOCAB)

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("Respeecher/ukrainian-data2vec")
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

model = Data2VecAudioForCTC.from_pretrained(
    "Respeecher/ukrainian-data2vec",
    vocab_size=len(vocab_dict),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
)

model.freeze_feature_encoder()

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Data2VecAudioForCTC LOAD REPORT from: Respeecher/ukrainian-data2vec
Key            | Status  | 
---------------+---------+-
lm_head.bias   | MISSING | 
lm_head.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [34]:
vocab = processor.tokenizer.get_vocab()
print(len(vocab))
print(sorted(vocab.items(), key=lambda x: x[1]))

encoded = processor.tokenizer(toronto_test_df["label"].iloc[0])
print(processor.tokenizer.convert_ids_to_tokens(encoded.input_ids))

40
[('|', 0), ("'", 1), ('а', 2), ('б', 3), ('в', 4), ('г', 5), ('д', 6), ('е', 7), ('ж', 8), ('з', 9), ('и', 10), ('й', 11), ('к', 12), ('л', 13), ('м', 14), ('н', 15), ('о', 16), ('п', 17), ('р', 18), ('с', 19), ('т', 20), ('у', 21), ('ф', 22), ('х', 23), ('ц', 24), ('ч', 25), ('ш', 26), ('щ', 27), ('ь', 28), ('ю', 29), ('я', 30), ('є', 31), ('і', 32), ('ї', 33), ('ґ', 34), ('’', 35), ('<unk>', 36), ('<pad>', 37), ('<s>', 38), ('</s>', 39)]
['з', 'а', 'р', 'о', 'д', 'и', 'л', 'и', 'с', 'ь', '|', 'п', 'р', 'о', 'т', 'о', 'ж', 'а', 'р', 'т', 'и', '|', 'п', 'р', 'о', '|', 'с', 'х', 'о', 'ж', 'і', 'с', 'т', 'ь', '|', 'з', 'в', 'у', 'к', 'і', 'в', '|', 'я', 'к', 'і', '|', 'б', 'у', 'л', 'и', '|', 'о', 'з', 'н', 'а', 'к', 'о', 'ю', '|', 'р', 'о', 'з', 'в', 'и', 'т', 'к', 'у', '|', 'м', 'о', 'в', 'л', 'е', 'н', 'н', 'я', '|', 'т', 'а', '|', 'п', 'и', 'с', 'е', 'м', 'н', 'о', 'с', 'т', 'і']


In [36]:
collator = DataCollatorCTCWithPadding(processor=processor)

toronto_train_ds = TorontoDataset(toronto_train_df, processor, TORONTO, features_column="input_values")
toronto_val_ds = TorontoDataset(toronto_test_df, processor, TORONTO, features_column="input_values")

toronto_train_dl = DataLoader(toronto_train_ds, batch_size=8, shuffle=True, collate_fn=collator)
toronto_val_dl = DataLoader(toronto_val_ds, batch_size=8, shuffle=False, collate_fn=collator)

logger = CSVLogger("logs", name="data2vec_finetune_val")
m = Data2VecCTCBaseline(model, processor)

trainer = L.Trainer(accelerator=device, logger=logger)
trainer.fit(m, train_dataloaders=toronto_train_dl, val_dataloaders=toronto_val_dl)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/alegator1209/micromamba/envs/deep-audio/lib/python3.11/site-packages/lightning/pytorch/loops/utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                ┃ Params ┃ Mode ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━┩
│ 0 │ model │ Data2VecAudioForCTC │  313 M │ eval │     0 │
└───┴───────┴─────────────────────┴────────┴──────┴───────┘

Trainable params: 309 M                                                                                            
Non-trainable params: 4.2 M                                                                                        
Total params: 313 M                                                                                                
Total estimated model params size (MB): 1,253.261                                                                  
Modules in train mode: 0                                                                                           
Modules in eval mode: 429                                                                                          
Total FLOPs: 0

Output()

/Users/alegator1209/micromamba/envs/deep-audio/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:
21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/Users/alegator1209/micromamba/envs/deep-audio/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/da
ta_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing
the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.

NotImplementedError: The operator 'aten::_ctc_loss' is not currently implemented for the MPS device. If you want this op to be considered for addition please comment on https://github.com/pytorch/pytorch/issues/141287 and mention use-case, that resulted in missing op as well as commit hash Unknown. As a temporary fix, you can set the environment variable `PYTORCH_ENABLE_MPS_FALLBACK=1` to use the CPU as a fallback for this op. WARNING: this will be slower than running natively on MPS.